# From a material recipe to a diffraction observation

This notebook demonstrates the `xray_generator` package end to end:

**crystals + texture + compact material → phase response → moving exposure → clean expectation → photon counts → polar view → reproducible files.**

This example simulates the scan of a Cu/Ti interface. The same material and representative orientations
are reused throughout, with the mixture we observe changing with beam trajectory.
Counting noise is added only after integration.

The default has 256 × 256 pixels and 512 representative orientations per phase.
Generated files go into `datasets/notebook_demo/` in a checkout.


In [ ]:
from pathlib import Path
from dataclasses import asdict, replace
from contextlib import redirect_stdout
from io import StringIO
import html
import json
import sys
import time

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from IPython.display import display, HTML

%matplotlib inline

CHECKOUT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / 'src/xray_generator').is_dir() and (p / 'pyproject.toml').is_file()), None)
if CHECKOUT is not None:
    sys.path.insert(0, str(CHECKOUT))
    sys.path.insert(0, str(CHECKOUT / 'src'))

import xray_generator
from xray_generator.api import Exposure, SimulationRequest, expected, observe, validate
from xray_generator.cli import main as simulate_cli, load_requests
from xray_generator.experiment.cache import ExpectationCache
from xray_generator.experiment.bundles import inspect_bundle
from xray_generator.experiment.provenance import fingerprint, filesystem_path
from xray_generator.material import ProceduralMaterial
from xray_generator.model import MaterialRecipe, PhaseDefinition, FractionRegion
from xray_generator.measurement.integrate import IntegrationOptions
from xray_generator.measurement.readout import poisson_readout
from xray_generator.physics.crystallography import built_in_phase, enumerate_reflections
from xray_generator.physics.detector import PlanarDetector
from xray_generator.physics.texture import TextureConfig, orientation_ensemble
from xray_generator.solvers.crystallographic import CrystallographicRateBackend, PhaseScattering
from xray_generator.preprocessing.polar import native_to_polar_bins, partition_counts

plt.rcParams.update({'figure.dpi': 110, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})

def table(rows):
    """Small readable tables without an extra dataframe dependency."""
    columns = list(rows[0])
    def fmt(value):
        return f'{value:,.6g}' if isinstance(value, (float, np.floating)) else str(value)
    head = ''.join(f'<th>{html.escape(c)}</th>' for c in columns)
    body = ''.join('<tr>' + ''.join(f'<td>{html.escape(fmt(row[c]))}</td>' for c in columns) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))

OUTPUT_ROOT = filesystem_path((CHECKOUT or Path.cwd()) / 'datasets/notebook_demo')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Package {xray_generator.__version__} · outputs: datasets/notebook_demo/')


## 1. Define crystals and a compact material

A material is a recipe, not a stored high-resolution raster. Queries evaluate it
only where needed. The phase definitions below derive composition and density
from explicit unit cells and atomic sites, so the material and diffraction
descriptions agree.

Coordinates here are in millimeters. The slab and beam diagram define where
phase fractions are sampled; the **representative illuminated amount** is
specified separately when constructing the scattering backend.


In [ ]:
crystals = [built_in_phase('cu_fcc'), built_in_phase('ti_hcp')]
phases = tuple(PhaseDefinition(
    crystal.phase_id,
    tuple(crystal.elemental_mass_fractions),
    tuple(crystal.elemental_mass_fractions.values()),
    crystal.density_g_cm3,
    source_id='explicit_cell_neutral_IT92',
) for crystal in crystals)

recipe = MaterialRecipe.sharp_interface(
    bounds_mm=((-1., 1.), (-0.1, 0.1), (0., 0.001)),
    phases=phases, interface_x_mm=0., left_fraction=1., right_fraction=0.,
    realization_id='notebook-cu-ti-interface',
)
material = ProceduralMaterial(recipe)
table([{'Phase': c.phase_id, 'Cell volume (Å³)': c.cell_volume_A3,
        'Density (g/cm³)': c.density_g_cm3, 'Occupied atomic sites': len(c.sites)} for c in crystals])
print(f'Recipe storage: {len(json.dumps(recipe.to_dict()).encode()):,} bytes')

In [ ]:
x = np.linspace(-0.8, 0.8, 601)
points = np.column_stack((x, np.zeros_like(x), np.full_like(x, 0.0005)))
state = material.state_at(points)
reverse_state = material.state_at(points[::-1])
assert np.array_equal(state.phase_volume_fractions, reverse_state.phase_volume_fractions[::-1])
assert np.allclose(state.phase_mass_fractions.sum(axis=1), 1.)

fig, ax = plt.subplots(figsize=(9, 2.7), layout='constrained')
ax.plot(x, state.phase_volume_fractions[:, 0], label='Cu volume fraction', lw=2)
ax.plot(x, state.phase_volume_fractions[:, 1], label='Ti volume fraction', lw=2)
ax.axvline(0, color='0.4', ls=':', label='Interface')
ax.set(xlabel='Scan coordinate x (mm)', ylabel='Phase volume fraction', ylim=(-0.05, 1.05))
ax.legend(ncol=3, loc='upper center')
plt.show()

mixture = ProceduralMaterial(replace(recipe, regions=(FractionRegion((-1., 1.), (0.5, 0.5)),)))
mixed = mixture.state_at([[0., 0., 0.0005]])
table([{'Phase': name, 'Mass fraction': mixed.phase_mass_fractions[0, i],
        'Volume fraction used for diffraction': mixed.phase_volume_fractions[0, i]}
       for i, name in enumerate(material.phase_ids)])

## 2. Add texture and a detector

The Cu phase has a tilted fiber texture; Ti has a partial fiber. Both include
an isotropic component. Directions are **Cartesian vectors**, and quaternions
use scalar-first `wxyz`. The shared fixed pose maps sample `z` onto lab `x`.
The beam travels along lab `+x`; detector columns run along `y`, rows along `z`.

`N_ORIENTATIONS` controls a finite representation of an orientation distribution,
not material volume or a count of positioned grains. `coherent_domain_nm`
controls reciprocal peak width independently. All reflections belonging to one
representative orientation rotate together.


In [ ]:
N_PIXELS = 512
N_ORIENTATIONS = 512
WAVELENGTH_A = 0.5166
pose = (float(np.sqrt(0.5)), 0., float(np.sqrt(0.5)), 0.)

textures = [
    TextureConfig(kind='fiber', num_orientations=N_ORIENTATIONS, seed=1211,
                  crystal_direction=(1., 1., 1.), sample_direction=(0.5, 0., np.sqrt(0.75)),
                  spread_deg=14., isotropic_fraction=0.15, sample_to_lab_wxyz=pose),
    TextureConfig(kind='partial_fiber', num_orientations=N_ORIENTATIONS, seed=1212,
                  crystal_direction=(0., 0., 1.), sample_direction=(0., 0., 1.),
                  spread_deg=12., spin_center_deg=35., spin_width_deg=80.,
                  isotropic_fraction=0.15, sample_to_lab_wxyz=pose),
]
detector = PlanarDetector((N_PIXELS, N_PIXELS), pixel_pitch_mm=51.2 / N_PIXELS, distance_mm=60.)
backend = CrystallographicRateBackend(
    [PhaseScattering(c, t, coherent_domain_nm=size)
     for c, t, size in zip(crystals, textures, (8., 6.))],
    detector=detector, wavelength_A=WAVELENGTH_A,
    incident_flux_density_photons_s_um2=1e10,
    illuminated_volume_um3=1., background_rate_per_pixel=0.05,
    pixel_subdivision=6, comparison_subdivision=3, support_sigma=7.5,
)
accuracy = IntegrationOptions(atol=0.001, rtol=0.01)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), layout='constrained')
for ax, crystal, texture in zip(axes, crystals, textures):
    population = orientation_ensemble(texture)
    poles = population.lab_rotation_matrices @ np.asarray(texture.crystal_direction)
    scatter = ax.scatter(poles[:, 1], poles[:, 2], c=poles[:, 0], cmap='coolwarm', vmin=-1, vmax=1, s=8, alpha=0.7)
    ax.add_patch(plt.Circle((0, 0), 1, fill=False, color='0.7'))
    ax.set(title=crystal.phase_id, xlabel='Lab y component', ylabel='Lab z component',
           xlim=(-1.05, 1.05), ylim=(-1.05, 1.05), aspect='equal')
fig.suptitle('Selected pole directions')
fig.colorbar(scatter, ax=list(axes), label='Lab x component', shrink=0.75)
plt.show()


## 3. Render each phase once

The cell metric, atomic basis and neutral-atom form factors determine reflection
positions and relative strengths. A normalized 3D reciprocal-space Gaussian
is integrated over native detector pixel areas, including clipped tails from
peaks centered outside the panel.

These **pure-phase rates at equal supplied volume** are reusable. Material
volume fractions and exposure integration combine them later. The third basis
image is detector background, which remains when the beam misses the sample.
All detector images below share an absolute colour scale; no image is normalized
to an arbitrary photon budget.


In [ ]:
start = time.perf_counter()
rates = backend.basis_rates()
print(f'Prepared phase responses in {time.perf_counter() - start:.2f} s')
table([{'Phase': row['phase_id'], 'Orientations': row['orientation_count'],
        'Signed reflections': row['reflection_count'], 'Accepted rate (events/s)': row['rate_sum']}
       for row in backend.preparation_report])

pitch = detector.pixel_pitch_mm
br, bc = detector.beam_center_row_col
extent = [(-0.5-bc)*pitch, (N_PIXELS-0.5-bc)*pitch,
          (-0.5-br)*pitch, (N_PIXELS-0.5-br)*pitch]

def detector_images(arrays, titles, *, label, columns=2):
    """Shared log scale, fixed physical axes, and no per-image rescaling."""
    rows = int(np.ceil(len(arrays) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(5 * columns, 4.2 * rows),
                             squeeze=False, layout='constrained')
    positive = np.concatenate([a[a > 0] for a in arrays])
    upper = float(positive.max())
    norm = LogNorm(vmin=max(upper / 1e4, 1e-6), vmax=upper)
    cmap = plt.get_cmap('magma').copy()
    cmap.set_bad('#000004')  # Zero counts are dark, not bright missing pixels.
    for ax, array, title in zip(axes.ravel(), arrays, titles):
        image = ax.imshow(np.ma.masked_less_equal(array, 0), origin='lower', extent=extent,
                          cmap=cmap, norm=norm, interpolation='nearest')
        ax.set(title=title, xlabel='Detector y (mm)', ylabel='Detector z (mm)')
    for ax in axes.ravel()[len(arrays):]: ax.set_visible(False)
    fig.colorbar(image, ax=list(axes.ravel()[:len(arrays)]), label=label, shrink=0.8)
    return fig

detector_images(list(rates[:2]), ['Pure Cu response', 'Pure Ti response'], label='Expected events / s / pixel')
plt.show()


The patchiness above is fixed finite-orientation sampling. It is distinct from
Poisson counting noise, which has not been drawn yet. More representatives
generally fill the rings more densely without increasing the supplied amount;
the current sampler generates a different finite approximation when its size
changes, rather than appending grains. This notebook does not establish the
continuum orientation-population limit.

## 4. Move the footprint across the same sample

Three 0.1 s acquisitions share one material: a stationary left-side reference,
a scan through the interface, and a later overlapping scan. The rectangular
footprint averages the field at each time; the shutter-open exposure then
integrates that rate. Increasing beam width at fixed supplied illuminated volume
changes spatial averaging, not the amount normalization.


In [ ]:
exposures = [
    Exposure('stationary-left', 0., 0.1, center_at_zero_mm=-0.4, beam_width_mm=0.2),
    Exposure('crossing', 0., 0.1, center_at_zero_mm=-0.4, velocity_mm_s=8., beam_width_mm=0.2),
    Exposure('overlap', 0.05, 0.15, center_at_zero_mm=-0.4, velocity_mm_s=8., beam_width_mm=0.2),
]
requests = [SimulationRequest(material, backend, exposure, accuracy) for exposure in exposures]
capabilities = validate(requests[0])
print(f"Profile: {capabilities['profile']} · experimentally calibrated: {capabilities['calibrated_physics']}")

fig, ax = plt.subplots(figsize=(9, 3.7), layout='constrained')
ax.axvspan(-1, 0, color='tab:orange', alpha=0.12, label='Cu region')
ax.axvspan(0, 1, color='tab:blue', alpha=0.12, label='Ti region')
for e, colour in zip(exposures, ('tab:green', 'black', 'tab:purple')):
    t = np.linspace(e.start_s, e.stop_s, 200)
    center = e.center_at_zero_mm + e.velocity_mm_s * t
    ax.plot(center, t, color=colour, label=e.exposure_id)
    ax.fill_betweenx(t, center-e.beam_width_mm/2, center+e.beam_width_mm/2, color=colour, alpha=0.14)
ax.set(xlabel='Beam position x (mm)', ylabel='Time (s)', title='Line = beam centre; band = footprint', xlim=(-0.6, 1.))
ax.legend(loc='upper left', ncol=2)
plt.show()


## 5. Integrate expectations and reuse the cache

`expected` integrates a few nonnegative mixture coefficients instead of
re-rendering the entire detector at every spatial/time node. Known sample
interfaces split the adaptive integration. The cache stores clean means and
their numerical error estimates, never noisy images.

On the first run the **acquisition** calls are cold; reruns may already hit
the disk cache. Phase responses were prepared above. The warm call and disk reload use zero new integration
evaluations. A successful tolerance check is numerical consistency, not a bound
on the missing experimental physics.


In [ ]:
cache = ExpectationCache(32 * 1024**2, disk_directory=OUTPUT_ROOT / 'expectation_cache',
                         max_disk_bytes=64 * 1024**2)
start = time.perf_counter()
means = [expected(request, cache=cache) for request in requests]
print(f'Acquisition expectations: {time.perf_counter() - start:.3f} s (phase render excluded)')
warm = expected(requests[1], cache=cache)
disk_cache = ExpectationCache(0, disk_directory=OUTPUT_ROOT / 'expectation_cache', max_disk_bytes=64 * 1024**2)
restored = expected(requests[1], cache=disk_cache)
assert warm.cache_hit and restored.cache_hit and warm.evaluations == restored.evaluations == 0
assert np.array_equal(warm.total_mean, restored.total_mean)
table([{'Exposure': e.exposure_id, 'Expected photons': float(m.total_mean.sum()),
        'Cache hit': m.cache_hit, 'Integration evaluations': m.evaluations,
        'Max error / allowed tolerance': float(np.max(m.numerical_error / (accuracy.atol + accuracy.rtol*m.total_mean)))}
       for e, m in zip(exposures, means)])
assert all(np.all(m.numerical_error <= accuracy.atol + accuracy.rtol*m.total_mean) for m in means)
table([{'Cache': 'Warm memory', 'Hit': warm.cache_hit, 'New evaluations': warm.evaluations},
       {'Cache': 'Disk reload', 'Hit': restored.cache_hit, 'New evaluations': restored.evaluations}])


### Check the moving-boundary integral independently

For this pure sharp interface, the instantaneous Cu footprint fraction is
`clip((interface − left_edge) / width, 0, 1)`. It is piecewise linear in time,
so trapezoids split at the edge-crossing times integrate it exactly. This check
uses the prepared response images but independent acquisition logic.


In [ ]:
def exact_cu_open_time(e):
    breaks = [e.start_s, e.stop_s]
    if e.velocity_mm_s:
        for center_at_edge in (-e.beam_width_mm/2, e.beam_width_mm/2):
            t = (center_at_edge-e.center_at_zero_mm) / e.velocity_mm_s
            if e.start_s < t < e.stop_s: breaks.append(t)
    t = np.array(sorted(breaks))
    left = e.center_at_zero_mm + e.velocity_mm_s*t - e.beam_width_mm/2
    fraction = np.clip(-left/e.beam_width_mm, 0., 1.)
    return float(np.sum(np.diff(t)*(fraction[:-1]+fraction[1:])/2) * e.shutter_fraction)

checks = []
for e, result in zip(exposures, means):
    cu_time = exact_cu_open_time(e)
    open_time = (e.stop_s-e.start_s)*e.shutter_fraction
    analytic_mean = cu_time*rates[0] + (open_time-cu_time)*rates[1] + open_time*rates[2]
    np.testing.assert_allclose(result.total_mean, analytic_mean, rtol=1e-11, atol=1e-10)
    checks.append({'Exposure': e.exposure_id, 'Cu effective open time (s)': cu_time,
                   'Ti effective open time (s)': open_time-cu_time})
table(checks)

# Doubling a stationary exposure doubles its expectation, without rescaling an image.
double_time = expected(replace(requests[0], exposure=replace(exposures[0], exposure_id='double-duration', stop_s=0.2)), cache=cache)
np.testing.assert_allclose(double_time.total_mean, 2*means[0].total_mean, rtol=1e-11, atol=1e-10)
print('Independent motion check and stationary dose scaling passed.')


## 6. Draw observations after integration

Changing `readout_id` gives a new conditional Poisson realization. Reusing the
same seed, exposure and readout identity reproduces the counts exactly. Neither
operation redraws the material or representative orientations.


In [ ]:
ROOT_SEED = 1729
observations = [observe(request, root_seed=ROOT_SEED, readout_id='A', cache=cache)[0] for request in requests]
replica_b, _ = observe(requests[1], root_seed=ROOT_SEED, readout_id='B', cache=cache)
replay_a, _ = observe(requests[1], root_seed=ROOT_SEED, readout_id='A', cache=cache)
assert np.array_equal(replay_a.counts, observations[1].counts)
assert all(o.counts.dtype.kind in 'iu' and np.all(o.counts >= 0) for o in observations)
table([{'Exposure': e.exposure_id, 'Expected photons': float(m.total_mean.sum()),
        'Observed photons (A)': int(o.counts.sum())} for e, m, o in zip(exposures, means, observations)])

fig = detector_images(
    [means[1].total_mean, observations[1].counts, means[2].total_mean, observations[2].counts],
    ['Crossing: clean expectation', 'Crossing: observed counts A',
     'Overlapping scan: clean expectation', 'Overlapping scan: observed counts A'],
    label='Photons per native pixel (common logarithmic scale)',
)
fig.savefig(OUTPUT_ROOT / 'expectations_and_counts.png', dpi=150)
plt.show()


In [ ]:
# Validate a count distribution at one fixed mean, not variance across pixels
# with different means. This small ensemble uses the native readout function.
mu = float(means[1].total_mean.ravel()[np.argmin(abs(means[1].total_mean.ravel()-10.))])
draws = poisson_readout(np.full(5000, mu), root_seed=ROOT_SEED, readout_id='fixed-mean-noise-check')
from scipy.stats import poisson
k = np.arange(max(0, int(draws.min())-1), int(draws.max())+2)
fig, axes = plt.subplots(1, 2, figsize=(10, 3), layout='constrained')
axes[0].hist(draws, bins=np.arange(k[0]-0.5, k[-1]+1.5), density=True, alpha=0.7, label='5,000 draws')
axes[0].plot(k, poisson.pmf(k, mu), 'o-', ms=3, label='Poisson probability')
axes[0].set(xlabel='Counts at a fixed mean', ylabel='Probability'); axes[0].legend()
axes[1].hist((observations[1].counts-replica_b.counts).ravel(), bins=61, color='tab:purple')
axes[1].set(xlabel='Count A − count B at the same native pixel', ylabel='Number of pixels',
            title='Same mean image, independent noise')
plt.show()
table([{'Predicted mean = variance': mu, 'Ensemble mean': float(draws.mean()),
        'Ensemble variance': float(draws.var())}])


## 7. Rebin counts without fractional photons

Native pixels are assigned whole to polar bins. Here **q = 2π|G|** in radians
per angstrom; it differs from the cycles-per-angstrom vectors used internally.
The centre ray sets q/azimuth for bin assignment. This is not subpixel
resampling. The full q range retains every pixel, and the sum must be exact.

An analysis mask hides part of the detector without changing its original mean
or redrawing its counts. Coverage records the fraction of eligible pixel area
retained in each bin; empty bins stay finite.


In [ ]:
row, col = np.indices(detector.shape)
g, solid_angle, direction = detector.sample_rays(row, col, wavelength_A=WAVELENGTH_A)
q = 2*np.pi*np.linalg.norm(g, axis=-1)
chi = np.arctan2(row-br, col-bc)
q_edges = np.linspace(0., np.nextafter(float(q.max()), np.inf), 97)
chi_edges = np.linspace(-np.pi, np.pi, 73)
bin_ids = native_to_polar_bins(q, chi, q_edges, chi_edges)
raw = observations[1].counts
polar = partition_counts(raw, bin_ids, 96*72)
valid = np.ones(detector.shape, dtype=bool)
valid[:, N_PIXELS//2-5:N_PIXELS//2+5] = False
masked = partition_counts(raw, bin_ids, 96*72, valid_mask=valid)
assert int(polar.counts.sum()) == int(raw.sum())
assert int(masked.counts.sum()) == int(raw[valid].sum())

fig, axes = plt.subplots(1, 3, figsize=(13, 3.3), layout='constrained')
polar_cmap = plt.get_cmap('magma').copy()
polar_cmap.set_bad('#000004')
image = axes[0].pcolormesh(q_edges, np.rad2deg(chi_edges), polar.counts.reshape(96, 72).T,
                          shading='flat', cmap=polar_cmap, norm=LogNorm(vmin=1, vmax=max(2, int(polar.counts.max()))))
axes[0].set(xlabel='q (rad/Å)', ylabel='Azimuth (degrees)', title='Whole-pixel polar counts')
fig.colorbar(image, ax=axes[0], label='Counts per bin')
axes[1].plot((q_edges[:-1]+q_edges[1:])/2, polar.counts.reshape(96, 72).sum(axis=1), label='Full detector')
axes[1].plot((q_edges[:-1]+q_edges[1:])/2, masked.counts.reshape(96, 72).sum(axis=1), label='Masked')
axes[1].set(xlabel='q (rad/Å)', ylabel='Counts per q bin', title='Radial sums (no area normalization)'); axes[1].legend()
cover = axes[2].pcolormesh(q_edges, np.rad2deg(chi_edges), masked.coverage.reshape(96, 72).T,
                          shading='flat', vmin=0, vmax=1, cmap='viridis')
axes[2].set(xlabel='q (rad/Å)', ylabel='Azimuth (degrees)', title='Coverage after masking')
fig.colorbar(cover, ax=axes[2], label='Retained fraction of eligible pixel area')
plt.show()
print(f'Native = polar: {int(raw.sum()):,} counts; after mask: {int(masked.counts.sum()):,}')


## 8. Export, inspect and recover the experiment

The same Python objects serialize to a CLI configuration. `observe` writes
integer observations, private clean truth, checksums and one shared recoverable
run record. The CLI below uses the same disk cache, so exporting should not
render the diffraction patterns again. Repeating it verifies and reuses the
completed bundles.

`observable.npz` is the observation. Keep `private.npz` and the material/scattering
truth out of a model's inputs when evaluating inference. Retain `.runs/` with
the observation folders so that inspection and recovery remain possible.


In [ ]:
config = {
    'schema_version': 'xray_generator.crystallographic_experiment/1',
    'profile': backend.profile, 'material': recipe.to_dict(),
    'crystallography': backend.to_dict(), 'exposures': [asdict(e) for e in exposures],
    'integration': asdict(accuracy), 'cache_memory_bytes': 32*1024**2,
    'readout': {'root_seed': ROOT_SEED, 'replicate_ids': ['A', 'B']},
}
RUN_DIR = OUTPUT_ROOT / fingerprint(config)[:16]
RUN_DIR.mkdir(parents=True, exist_ok=True)
config_path = RUN_DIR / 'experiment.json'
config_path.write_text(json.dumps(config, indent=2), encoding='utf-8')

def cli_json(arguments):
    capture = StringIO()
    with redirect_stdout(capture):
        status = simulate_cli(arguments)
    assert status == 0
    return json.loads(capture.getvalue())

arguments = ['observe', str(config_path), '--output', str(RUN_DIR / 'observations'),
             '--cache-dir', str(OUTPUT_ROOT / 'expectation_cache'), '--cache-disk-bytes', str(64*1024**2)]
saved = cli_json(arguments)
reused = cli_json(arguments)
assert all(item['cache_hit'] for item in saved['outputs'])
assert all(item['cache_hit'] and item['reused'] for item in reused['outputs'])
table([{'Exposure': item['exposure_id'], 'Readout': item['readout_id'],
        'Observed photons': item['counts'], 'Expectation reused': item['cache_hit']}
       for item in saved['outputs']])
print('Six observation bundles written; replay verified all six.')


In [ ]:
chosen = next(item for item in saved['outputs'] if item['exposure_id']=='crossing' and item['readout_id']=='A')
bundle = filesystem_path(chosen['path'])
manifest = inspect_bundle(bundle)
with np.load(bundle / 'observable.npz', allow_pickle=False) as public:
    np.testing.assert_array_equal(public['counts'], observations[1].counts)
    print('Observable fields:', public.files)
with np.load(bundle / 'private.npz', allow_pickle=False) as truth:
    np.testing.assert_array_equal(truth['expected_mean'], means[1].total_mean)
    print('Private truth fields:', truth.files)

# Use the returned record ID, not a glob that could pick an older run.
record_path = filesystem_path(RUN_DIR / 'observations/.runs' / (saved['run_record_id'] + '.json'))
record = json.loads(record_path.read_text(encoding='utf-8'))
recovered = ProceduralMaterial(MaterialRecipe.from_dict(record['configuration']['material']))
assert recovered.fingerprint == material.fingerprint
recovered_config_path = RUN_DIR / 'recovered_experiment.json'
recovered_config_path.write_text(json.dumps(record['configuration'], indent=2), encoding='utf-8')
loaded_config, loaded_requests = load_requests(recovered_config_path)
assert loaded_requests[1].backend.fingerprint == backend.fingerprint
assert loaded_requests[1].exposure == exposures[1]
reloaded = expected(loaded_requests[1], cache=disk_cache)
assert reloaded.cache_hit
np.testing.assert_array_equal(reloaded.total_mean, means[1].total_mean)
table([{'Check': 'Bundle checksums and run record', 'Passed': manifest['status']=='complete'},
       {'Check': 'Recovered compact material identity', 'Passed': recovered.fingerprint==material.fingerprint},
       {'Check': 'Full experiment recovered from run record + disk cache', 'Passed': reloaded.cache_hit}])
print('Artifacts: datasets/notebook_demo/<configuration fingerprint>/')


## 9. Optional extensions

The main walkthrough above is complete. The next two small examples illustrate
other provider capabilities without claiming additional physical transport.

### Reference rates as an alternative backend

A supplied-rate model can exercise material, acquisition and readout without
crystallographic rendering. The two native pixels below have arbitrary rates, not
an XRF spectrum or calibrated reference measurements.


In [ ]:
from xray_generator.solvers.reference import ReferenceRateBackend
reference_rates = np.array([[120., 20.], [30., 160.]])[:, None, :]  # phase, row, column
reference_backend = ReferenceRateBackend(reference_rates, material.phase_ids)
reference_request = SimulationRequest(material, reference_backend, exposures[1], accuracy)
reference_observation, reference_mean = observe(reference_request, root_seed=ROOT_SEED)
table([{'Synthetic pixel': i, 'Expected events': float(mean), 'Observed events': int(count)}
       for i, (mean, count) in enumerate(zip(reference_mean.total_mean.ravel(), reference_observation.counts.ravel()))])


### Lazy identities in a bounded geometry fixture

This separate provider generates only the columnar cells intersecting a query
window, with reproducible identities and orientations. It supports pure,
constant-phase regions such as our recipe. It is a geometry/identity fixture,
**not a morphology model, and it is not connected to the diffraction backend**.
The identity demonstration must not be interpreted as grain-overlap correlations
in the diffraction images above.


In [ ]:
from xray_generator.material.lazy_grains import LazyGrainGenerator
grains = LazyGrainGenerator(material, cell_width_mm=0.08, seed=5)
left_window = ((-0.25, 0.12), (-0.1, 0.1), (0., 0.001))
right_window = ((-0.04, 0.33), (-0.1, 0.1), (0., 0.001))
left_batches = list(grains.iter_grains(left_window, batch_size=3))
right_batches = list(grains.iter_grains(right_window, batch_size=2))
left_ids = {identity: q for batch in left_batches for identity, q in zip(batch.grain_ids, batch.orientations_wxyz)}
right_ids = {identity: q for batch in right_batches for identity, q in zip(batch.grain_ids, batch.orientations_wxyz)}
shared = sorted(left_ids.keys() & right_ids.keys())
assert shared and all(np.array_equal(left_ids[key], right_ids[key]) for key in shared)
table([{'Window': 'Left query', 'Generated cells': len(left_ids)},
       {'Window': 'Right query', 'Generated cells': len(right_ids)},
       {'Window': 'Shared IDs with identical orientation', 'Generated cells': len(shared)}])
